<a href="https://colab.research.google.com/github/chin1230/Light-them-all/blob/main/embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Language Word Embeddings

In class, we discussed how we can reduce the dimensionality of word representations from their original vector space to an embedding space on the order of a few hundred dimensions. Different modeling choices for word embeddings may be ultimately evaluated by the effectiveness of classification or retrieval models.

In this assignment, however, we will consider another common method of evaluating word embeddings: by judging the usefulness of pairwise distances between words in the embedding space.

Follow along with the examples in this notebook, and implement the sections of code flagged with **TODO**.

In [17]:
## If you already have gensim, you can comment this out.
!pip install gensim

In [18]:
import gensim
import numpy as np
from gensim.test.utils import datapath, get_tmpfile
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

We'll start by downloading a plain-text version of the plays of William Shakespeare.

In [19]:
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/shakespeare_plays.txt
lines = [s.split() for s in open('shakespeare_plays.txt')]

--2026-11-25 04:10:49--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/shakespeare_plays.txt
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4746840 (4.5M) [text/plain]
Saving to: ‘shakespeare_plays.txt.1’

shakespeare_plays.t 100%[===================>]   4.53M  29.1MB/s    in 0.2s    

2026-11-25 04:10:50 (29.1 MB/s) - ‘shakespeare_plays.txt.1’ saved [4746840/4746840]



Then, we'll estimate a simple word2vec model on the Shakespeare texts.

In [20]:
model = Word2Vec(lines)

Even with such a small training set size, you can perform some standard analogy tasks we discussed in class.

In [21]:
model.wv.most_similar(positive=['king','woman'], negative=['man'])

[('queen', 0.766964316368103),
 ('warwick', 0.723652720451355),
 ('prince', 0.7157737612724304),
 ('clarence', 0.7093199491500854),
 ('york', 0.6837248802185059),
 ('duke', 0.677722156047821),
 ('gloucester', 0.6686105132102966),
 ('percy', 0.6609146595001221),
 ('son', 0.6538910865783691),
 ('cardinal', 0.65021812915802)]

In other words, we want a vector close to `king` and `woman` but subtracting the dimensions that are important to `man`, i.e., `queen`. Other words are mostly noble titles important in Shakespeare's "history" plays.

For the rest of this assignment, we will focus on finding words with similar embeddings, both within and across languages. For example, what words are similar to the name of the title character of *Othello*?

In [22]:
model.wv.most_similar(positive=['othello'])
#model.wv.most_similar(positive=['brutus'])

[('desdemona', 0.965722382068634),
 ('emilia', 0.9408660531044006),
 ('cleopatra', 0.9145029783248901),
 ('countess', 0.9117713570594788),
 ('iago', 0.9025204181671143),
 ('cassio', 0.90132075548172),
 ('imogen', 0.8805630803108215),
 ('pedro', 0.8787645697593689),
 ('proteus', 0.8784574270248413),
 ('petruchio', 0.8705960512161255)]

If you know the play, you might see some familiar names.

This search uses cosine similarity. In the default API, you should see the same similarity between the words `othello` and `desdemona` as in the search results above.

In [23]:
model.wv.similarity('othello', 'desdemona')

np.float32(0.96572244)

**TODO**: Your **first task**, therefore, is to implement your own cosine similarity function so that you can reuse it outside of the context of the gensim model object.

In [24]:
## TODO: Implement cosim
def cosim(v1, v2):
  ## return cosine similarity between v1 and v2
  dot_product = np.dot(v1, v2)
  return dot_product / (np.linalg.norm(v1) * np.linalg.norm(v2))

## This should give a result similar to model.wv.similarity:
cosim(model.wv['othello'], model.wv['desdemona'])

np.float32(0.96572256)

## Evaluation

We could collect a lot of human judgments about how similar pairs of words, or pairs of Shakespearean characters, are. Then we could compare different word-embedding models by their ability to replicate these human judgments.

If we extend our ambition to multiple languages, however, we can use a word translation task to evaluate word embeddings.

We will use a subset of [Facebook AI's FastText cross-language embeddings](https://fasttext.cc/docs/en/aligned-vectors.html) for several languages. Your task will be to compare English both to French, and to *one more language* from the following set: Arabic, German, Portuguese, Russian, Spanish, Vietnamese, and Chinese.

In [25]:
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.en.vec
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.fr.vec

# TODO: uncomment at least one of these to work with another language
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.ar.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.de.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.pt.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.ru.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.es.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.vi.vec
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.zh.vec

--2026-11-25 04:11:05--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.en.vec
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67681172 (65M)
Saving to: ‘30k.en.vec.1’

30k.en.vec.1        100%[===================>]  64.54M   131MB/s    in 0.5s    

2026-11-25 04:11:05 (131 MB/s) - ‘30k.en.vec.1’ saved [67681172/67681172]

--2026-11-25 04:11:05--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.fr.vec
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67802327 (65M)
Saving to: ‘30k.fr.vec.1’

30k.fr.vec.1        100%[===================>]  64.66M   133MB/s    in 0.5s    

2026-11-25 04:11:06 (133 MB/s) - ‘30k.fr.vec.1’ saved [67802327/67802327]

--2026-11-25 04:11:06--  http://www.cc

We'll start by loading the word vectors from their textual file format to a dictionary mapping words to numpy arrays.

In [26]:
def vecref(s):
  (word, srec) = s.split(' ', 1)
  return (word, np.fromstring(srec, sep=' '))

def ftvectors(fname):
  return { k:v for (k, v) in [vecref(s) for s in open(fname)] if len(v) > 1}

envec = ftvectors('30k.en.vec')
frvec = ftvectors('30k.fr.vec')

# TODO: load vectors for one more language, such as zhvec (Chinese)
# arvec = ftvectors('30k.ar.vec')
# devec = ftvectors('30k.de.vec')
# ptvec = ftvectors('30k.pt.vec')
# ruvec = ftvectors('30k.ru.vec')
# esvec = ftvectors('30k.es.vec')
# vivec = ftvectors('30k.vi.vec')
zhvec = ftvectors('30k.zh.vec')

**TODO**: Your next task is to write a simple function that takes a vector and a dictionary of vectors and finds the most similar item in the dictionary. For this part of the assignment, a linear scan through the dictionary using your `cosim` function from above is acceptible.

In [27]:
## TODO: implement this search function
def mostSimilar(vec, vecDict):
  ## Use your cosim function from above
  best_word = ''
  best_score = float('-inf')

  for word, word_vec in vecDict.items():
    score = cosim(vec, word_vec)
    if score > best_score:
      best_word = word
      best_score = score

  return (best_word, best_score)

## some example searches
[mostSimilar(envec[e], frvec) for e in ['computer', 'germany', 'matrix', 'physics', 'yeast']]

[('informatique', np.float64(0.5023827767603765)),
 ('allemagne', np.float64(0.593718413875964)),
 ('matrice', np.float64(0.5088361302065517)),
 ('physique', np.float64(0.4555543434796394)),
 ('fermentation', np.float64(0.3504105196166514))]

Some matches make more sense than others. Note that `computer` most closely matches `informatique`, the French term for *computer science*. If you looked further down the list, you would see `ordinateur`, the term for *computer*. This is one weakness of a focus only on embeddings for word *types* independent of context.

To evalute cross-language embeddings more broadly, we'll look at a dataset of links between Wikipedia articles.

In [28]:
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/links.tab
links = [s.split() for s in open('links.tab')]

--2026-11-25 04:11:13--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/links.tab
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1408915 (1.3M)
Saving to: ‘links.tab.1’

links.tab.1         100%[===================>]   1.34M  --.-KB/s    in 0.1s    

2026-11-25 04:11:14 (12.0 MB/s) - ‘links.tab.1’ saved [1408915/1408915]



This `links` variable consists of triples of `(English term, language, term in that language)`. For example, here is the link between English `academy` and French `académie`:

In [29]:
links[302]

['academy', 'fr', 'académie']

We construct a test set for English-French from the first 1000 links between those languages.

In [30]:
frtest = [x for x in links if x[1] == "fr"][0:1000]
frtest[0:10]

[['aalborg', 'fr', 'aalborg'],
 ['aarhus', 'fr', 'aarhus'],
 ['aba', 'fr', 'aba'],
 ['abad', 'fr', 'abad'],
 ['abandon', 'fr', 'abandon'],
 ['abbas', 'fr', 'abbas'],
 ['abbasid', 'fr', 'abbassides'],
 ['abbess', 'fr', 'abbesse'],
 ['abbey', 'fr', 'abbaye'],
 ['abbot', 'fr', 'abbé']]

**TODO**: Evaluate the English and French embeddings by computing the proportion of English Wikipedia articles whose corresponding French article in this test set `frtest` is also the closest word in embedding space. Skip English articles not covered by the word embedding dictionary.

Since many articles, e.g., about named entities, have the same title in English and French, use the identity function as a baseline and compute its accuracy. In other words, how often would you find the right French articles by simply echoing the English title as if it were French? In the ten example links above, `aalborg` and `aarhus` (two cities in Denmark) are the same in English and French. Remember to iterate only over the 1000 linked Wikipedia articles in the test set, not the entire embedding dictionary.

In [31]:
## TODO: Compute English-French Wikipedia retrieval accuracy.
correct = 0
baseline_correct = 0
total = 0

for en_word, _, fr_word in frtest:
    if en_word not in envec:
        continue

    total += 1

    predicted_word, score = mostSimilar(envec[en_word], frvec)
    if predicted_word == fr_word:
        correct += 1

    if en_word == fr_word:
        baseline_correct += 1

accuracy = correct / total
baselineAccuracy = baseline_correct / total

print(f"Embedding accuracy: {correct}/{total} = {accuracy:.2%}")
print(f"Baseline accuracy: {baseline_correct}/{total} = {baselineAccuracy:.2%}")

Embedding accuracy: 575/1000 = 57.50%
Baseline accuracy: 661/1000 = 66.10%


**TODO**: Compute the accuracy, i.e. precision@1, of the embeddings and of the identity function for the first 1000 links between English and another language besides French. Although the baseline will be lower for languages not written in the Roman alphabet (i.e., Arabic or Chinese), there are still many articles in those languages with headwords written in Roman characters.

In [32]:
## TODO: Compute English-X Wikipedia retrieval accuracy.
zhtest = [record for record in links if record[1] == 'zh'][:1000]

zh_correct = 0
zh_baseline_correct = 0
zh_total = 0

for en_word, _, zh_word in zhtest:
    if en_word not in envec:
        continue

    zh_total += 1
    predicted_word, _ = mostSimilar(envec[en_word], zhvec)

    if predicted_word == zh_word:
        zh_correct += 1

    if en_word == zh_word:
        zh_baseline_correct += 1

zh_accuracy = zh_correct / zh_total
zh_baseline_accuracy = zh_baseline_correct / zh_total

print(f"English-Chinese embedding accuracy: {zh_correct}/{zh_total} = {zh_accuracy:.2%}")
print(f"English-Chinese baseline accuracy: {zh_baseline_correct}/{zh_total} = {zh_baseline_accuracy:.2%}")

English-Chinese embedding accuracy: 124/1000 = 12.40%
English-Chinese baseline accuracy: 67/1000 = 6.70%


**TODO**: Find the 10 nearest neighbors of each English term to compute "recall at 10" and "mean reciprocal rank at 10".

In [33]:
## TODO: Compute recall@10 and MRR@10 when retrieving 10 nearest neighbors in French and some other language.
from heapq import nlargest

def evaluate_top10(test_set, target_vectors):
    hits = 0
    reciprocal_rank_sum = 0
    total = 0

    for en_word, _, correct_word in test_set:
        if en_word not in envec:
            continue

        total += 1
        query_vector = envec[en_word]

        top_words = nlargest(
            10,
            target_vectors,
            key=lambda word: cosim(query_vector, target_vectors[word])
        )

        if correct_word in top_words:
            hits += 1
            rank = top_words.index(correct_word) + 1
            reciprocal_rank_sum += 1 / rank

    recall_at_10 = hits / total
    mrr_at_10 = reciprocal_rank_sum / total
    return recall_at_10, mrr_at_10

fr_recall, fr_mrr = evaluate_top10(frtest, frvec)
zh_recall, zh_mrr = evaluate_top10(zhtest, zhvec)

print(f"English-French:  Recall@10 = {fr_recall:.2%}, MRR@10 = {fr_mrr:.4f}")
print(f"English-Chinese: Recall@10 = {zh_recall:.2%}, MRR@10 = {zh_mrr:.4f}")

English-French:  Recall@10 = 64.40%, MRR@10 = 0.6014
English-Chinese: Recall@10 = 21.60%, MRR@10 = 0.1536


## Speeding up Vector Search (required for CS6200, 20 points extra for CS4220)

The list of Wikipedia headwords is short enough that a linear scan through the non-English language embeddings takes some time but is feasible. In a production system, you could index the word embeddings using SimHash or some other locality sensitive hashing scheme, as we discussed for duplicate detection, to speed up this process.

A relatively easy way to get started with fast vector similarity search is to install Meta's `faiss` (Facebook AI Similarity Search) package and read [the tutorial](https://github.com/facebookresearch/faiss/wiki/Getting-started).

In [34]:
# Outside of colab, you may need a different package manager.
# !apt install libomp-dev
!pip install --upgrade faiss-cpu
# Use this line instead if you have a GPU.
# !python -m pip install --upgrade faiss-gpu
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 27.6 MB/s  0:00:00


**TODO**: Create two vector indexes, for the FastText embeddings of French and for the other language you evaluated above. Use `faiss` to find the 10 nearest neighbors for the top 1000 Wikipedia headwords you evaluated for English-French and the English-X as above.

In [35]:
## TODO: Create two FAISS indexes for the embeddings in French
##       and the other non-English language of your choice.


First, measure the _effectiveness_ of this approximate vector search approach. How does the R@10 and MRR@10 using `faiss` compare to the brute-force search you did above?

In [36]:
## TODO: Measure the R@10 and MRR@10 of the test set for the two languages.
##       Compute the difference in effecticeness to your brute-force search above.

Second, measure the _efficiency_ of this approach. How long in seconds does finding nearest neighbors for 1000 headwords by brute force compare to using `faiss`? (For this exercise, don't worry about amortizing indexing costs.)

In [37]:
## TODO: Find the 1000 nearest heawords in French and the other language for each query in the test set.
##       Compare the retrieval time in seconds to the speed of your brute-force search.
##       You shouldn't include the time to create the FAISS index.